# WiSARv1 Dataset Organization Investigation

This notebook performs a **read-only structural investigation** of a WiSARv1 dataset. It inventories paths, directory names, filenames, and metadata/manifests without opening image pixel data, modifying the dataset, copying files, extracting archives, resizing images, or training models.

The outputs are small aggregate text/CSV reports intended to answer whether defensible flight, recording-session, sequence, scene, or collection identifiers exist for leakage-safe splitting. Findings are labeled `documented`, `observed`, or `unknown`; different names are never treated as proof of different flights.

In [48]:
from __future__ import annotations

import csv
import os
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

# Set WISAR_DATASET_ROOT to the mounted dataset directory in Colab.
# Expected ZIP location: /content/drive/MyDrive/WiSARD/WiSARDv1.zip
DATASET_ROOT = Path(os.environ.get("WISAR_DATASET_ROOT", "/content/drive/MyDrive/WiSARD"))
if not DATASET_ROOT.exists() and Path("data/raw/WiSARD").is_dir():
    DATASET_ROOT = Path("data/raw/WiSARD")
REPORT_DIR = Path(os.environ.get("WISAR_REPORT_DIR", "results/dataset_audit"))
ZIP_NAME = "WiSARDv1.zip"
TREE_MAX_DEPTH = 4
MAX_METADATA_BYTES = 2_000_000
MAX_TEXT_LINES_PER_FILE = 2_000

METADATA_EXTENSIONS = {
    ".csv", ".json", ".xml", ".yaml", ".yml", ".txt", ".tsv", ".mat",
    ".ini", ".cfg", ".conf", ".toml", ".md", ".md5", ".log",
}
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp",
    ".gif", ".ppm", ".pgm", ".dng", ".heic",
}
SEARCH_TERMS = (
    "flight", "sequence", "session", "recording", "collection", "scene",
    "timestamp", "gps", "trajectory", "video", "mission",
)
GROUPING_TERMS = (
    "flight", "sequence", "session", "recording", "collection", "scene",
    "mission", "run", "take", "trip", "set",
)


In [45]:
def tokenize_name(value: str) -> list[str]:
    """Split names into stable lowercase alphanumeric tokens without opening files."""
    return [token for token in re.split(r"[^a-zA-Z0-9]+", value.lower()) if token]


In [49]:
if not DATASET_ROOT.exists() or not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Set DATASET_ROOT to the mounted WiSARv1 directory; not found: {DATASET_ROOT}"
    )

root_resolved = DATASET_ROOT.resolve()
file_records = []
directory_records = []
metadata_records = []
extension_counts = Counter()
extension_by_directory = Counter()
directory_file_counts = Counter()
name_tokens = Counter()
directory_name_tokens = Counter()
tree_lines = [f"{root_resolved.name}/"]

# os.scandir reads directory entries and stat information; it does not decode image pixels.
stack = [(root_resolved, 0)]
while stack:
    current, depth = stack.pop()
    try:
        entries = sorted(os.scandir(current), key=lambda entry: (not entry.is_dir(follow_symlinks=False), entry.name.lower()))
    except (OSError, PermissionError) as error:
        directory_records.append({"relative_directory": str(current.relative_to(root_resolved)), "status": f"unreadable: {error}"})
        continue

    relative_current = current.relative_to(root_resolved)
    directory_records.append({"relative_directory": "." if relative_current == Path(".") else str(relative_current), "status": "read"})
    if depth <= TREE_MAX_DEPTH:
        tree_lines.extend([f"{'  ' * (depth + 1)}{'[D] ' if entry.is_dir(follow_symlinks=False) else '[F] '}{entry.name}" for entry in entries])

    for entry in entries:
        entry_path = Path(entry.path)
        relative_path = entry_path.relative_to(root_resolved)
        if entry.is_dir(follow_symlinks=False):
            directory_name_tokens.update(tokenize_name(entry.name))
            stack.append((entry_path, depth + 1))
            continue
        if not entry.is_file(follow_symlinks=False):
            continue

        suffix = entry_path.suffix.lower() or "[no_extension]"
        relative_directory = str(relative_path.parent)
        extension_counts[suffix] += 1
        extension_by_directory[(relative_directory, suffix)] += 1
        directory_file_counts[relative_directory] += 1
        tokens = tokenize_name(entry.name)
        name_tokens.update(tokens)
        record = {
            "relative_path": str(relative_path),
            "relative_directory": relative_directory,
            "filename": entry.name,
            "extension": suffix,
            "size_bytes": entry.stat(follow_symlinks=False).st_size,
            "name_tokens": ",".join(tokens),
        }
        file_records.append(record)
        if suffix in METADATA_EXTENSIONS:
            metadata_records.append(record.copy())

print(f"Dataset root: {root_resolved}")
print(f"Directories observed: {len(directory_records):,}")
print(f"Files observed: {len(file_records):,}")
print(f"Metadata-like files: {len(metadata_records):,}")
print("No image file was opened as pixel data.")

Dataset root: D:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\data\raw\WiSARD
Directories observed: 1
Files observed: 1
Metadata-like files: 0
No image file was opened as pixel data.


In [50]:
zip_path = DATASET_ROOT / ZIP_NAME if DATASET_ROOT.is_dir() else None
zip_member_records = []
zip_metadata_matches = []
zip_metadata_snippets = []
zip_candidate_rows = []
zip_sensor_rows = []
zip_pattern_counts = Counter()
zip_pattern_examples = defaultdict(list)
zip_extension_counts = Counter()
zip_tree_lines = []
zip_status = "unknown"

if zip_path is not None and zip_path.is_file():
    zip_status = "observed"
    # ZipFile reads the archive directory and selected metadata members only; it never extracts files.
    with zipfile.ZipFile(zip_path, mode="r") as archive:
        infos = archive.infolist()
        member_pairs = [
            (info, info.filename.replace("\\", "/").strip("/"))
            for info in infos
            if info.filename.strip("/")
        ]
        zip_member_records = []
        for info, member_path in member_pairs:
            if not member_path:
                continue
            suffix = Path(member_path).suffix.lower() or "[no_extension]"
            is_directory = info.is_dir() or info.filename.endswith(("/", "\\"))
            if not is_directory:
                zip_extension_counts[suffix] += 1
            zip_member_records.append({
                "member_path": member_path,
                "relative_directory": str(Path(member_path).parent),
                "filename": Path(member_path).name,
                "extension": suffix,
                "size_bytes": info.file_size,
                "compressed_size_bytes": info.compress_size,
                "is_directory": is_directory,
            })

        tree_nodes = {"": {"directories": set(), "files": set()}}
        for record in zip_member_records:
            parts = Path(record["member_path"]).parts
            for index in range(len(parts)):
                parent = "/".join(parts[:index])
                node = parts[index]
                tree_nodes.setdefault(parent, {"directories": set(), "files": set()})
                if index < len(parts) - 1 or record["is_directory"]:
                    tree_nodes.setdefault(parent, {"directories": set(), "files": set()})["directories"].add(node)
                else:
                    tree_nodes.setdefault(parent, {"directories": set(), "files": set()})["files"].add(node)

        zip_tree_lines = [f"{DATASET_ROOT.name}/{ZIP_NAME}"]
        for parent, node in sorted(tree_nodes.items()):
            depth = 0 if not parent else len(Path(parent).parts)
            if depth > TREE_MAX_DEPTH:
                continue
            prefix = "  " * (depth + 1)
            for directory in sorted(node["directories"]):
                zip_tree_lines.append(f"{prefix}[D] {directory}")
            for filename in sorted(node["files"]):
                zip_tree_lines.append(f"{prefix}[F] {filename}")
else:
    zip_status = "unknown"

print(f"ZIP path: {zip_path}")
print(f"ZIP status: {zip_status}")
print(f"ZIP members observed: {len(zip_member_records):,}")
print("ZIP extracted: no; image pixels opened: no")

ZIP path: d:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\data\raw\WiSARD\WiSARDv1.zip
ZIP status: unknown
ZIP members observed: 0
ZIP extracted: no; image pixels opened: no


In [51]:
metadata_matches = []
metadata_snippets = []
pattern_counts = Counter()
pattern_examples = defaultdict(list)

for record in metadata_records:
    path = root_resolved / record["relative_path"]
    filename_lower = record["filename"].lower()
    filename_terms = [term for term in SEARCH_TERMS if term in filename_lower]
    content_terms = []
    content = ""
    content_status = "not_read"
    if record["extension"] != ".mat" and record["size_bytes"] <= MAX_METADATA_BYTES:
        try:
            content = path.read_text(encoding="utf-8", errors="replace")
            content_status = "read"
        except (OSError, UnicodeError) as error:
            content_status = f"unreadable: {error}"
    elif record["extension"] == ".mat":
        content_status = "binary_mat_not_decoded"
    else:
        content_status = "skipped_over_size_limit"

    if content:
        content_lower = content.lower()
        content_terms = [term for term in SEARCH_TERMS if term in content_lower]
        for line_number, line in enumerate(content.splitlines()[:MAX_TEXT_LINES_PER_FILE], start=1):
            line_terms = [term for term in SEARCH_TERMS if term in line.lower()]
            if line_terms and len(metadata_snippets) < 200:
                metadata_snippets.append({
                    "relative_path": record["relative_path"],
                    "line_number": line_number,
                    "terms": ",".join(line_terms),
                    "snippet": re.sub(r"\s+", " ", line.strip())[:240],
                })

    all_terms = sorted(set(filename_terms + content_terms))
    if all_terms:
        evidence_status = "documented" if content_terms else "observed"
        metadata_matches.append({
            "relative_path": record["relative_path"],
            "extension": record["extension"],
            "filename_terms": ",".join(filename_terms),
            "content_terms": ",".join(content_terms),
            "terms": ",".join(all_terms),
            "content_status": content_status,
            "evidence_status": evidence_status,
        })

for record in file_records:
    source_name = f"{record['relative_directory']}/{record['filename']}"
    for pattern, label in (
        (r"(?:^|[^a-zA-Z])(?:flight|sequence|session|recording|collection|scene|mission|run|take|trip|set)[-_]?[a-zA-Z0-9]+", "grouping_term_with_value"),
        (r"(?:^|[^a-zA-Z])\d{2,}(?:[^a-zA-Z]|$)", "numeric_identifier"),
        (r"[A-Za-z]+[_-]\d+", "label_number"),
    ):
        matches = re.findall(pattern, source_name, flags=re.IGNORECASE)
        if matches:
            pattern_counts[label] += len(matches)
            for match in matches[:3]:
                if len(pattern_examples[label]) < 10:
                    pattern_examples[label].append(match.strip(" _-"))

candidate_rows = []
for record in file_records:
    path_parts = Path(record["relative_path"]).parts
    for part in path_parts:
        part_lower = part.lower()
        matched_terms = [term for term in GROUPING_TERMS if term in part_lower]
        has_identifier_shape = bool(re.search(r"(?:^|[_-])(?:\d+|[a-z]+\d+)(?:$|[_-])", part_lower))
        if matched_terms or has_identifier_shape:
            candidate_rows.append({
                "candidate": part,
                "source_path": record["relative_path"],
                "matched_terms": ",".join(matched_terms),
                "evidence_status": "observed",
                "interpretation": "Naming/path pattern only; not proof of an independent flight or session.",
            })

# Canonicalize RGB/thermal paths only for comparison; this does not alter source paths.
sensor_groups = defaultdict(lambda: {"rgb": set(), "thermal": set()})
for record in file_records:
    parts = list(Path(record["relative_path"]).parts)
    sensors = {part.lower() for part in parts if part.lower() in {"rgb", "thermal", "visible", "infrared", "ir"}}
    if not sensors:
        continue
    sensor = "rgb" if "rgb" in sensors or "visible" in sensors else "thermal"
    canonical_parts = [part.lower() for part in parts if part.lower() not in {"rgb", "thermal", "visible", "infrared", "ir"}]
    canonical = "/".join(canonical_parts[:-1]) if canonical_parts else "."
    sensor_groups[canonical][sensor].add(record["filename"].lower())

sensor_rows = []
for canonical, groups in sorted(sensor_groups.items()):
    has_both = bool(groups["rgb"] and groups["thermal"])
    sensor_rows.append({
        "canonical_path_without_sensor": canonical,
        "rgb_file_count": len(groups["rgb"]),
        "thermal_file_count": len(groups["thermal"]),
        "shared_filename_count": len(groups["rgb"] & groups["thermal"]),
        "evidence_status": "observed" if has_both else "unknown",
        "interpretation": (
            "RGB and thermal occur under a shared canonical path; verify timestamps/metadata before grouping."
            if has_both else
            "No paired path observed here; this does not prove different flights or sessions."
        ),
    })

print(f"Metadata files matching investigation terms: {len(metadata_matches):,}")
print(f"Candidate grouping path/name observations: {len(candidate_rows):,}")
print(f"RGB/thermal canonical groups: {len(sensor_rows):,}")

Metadata files matching investigation terms: 0
Candidate grouping path/name observations: 0
RGB/thermal canonical groups: 0


In [34]:
if zip_status == "observed":
    with zipfile.ZipFile(zip_path, mode="r") as archive:
        info_by_path = {info.filename.replace("\\", "/").strip("/"): info for info in archive.infolist()}
        for record in zip_member_records:
            member_path = record["member_path"]
            path_lower = member_path.lower()
            filename_terms = [term for term in SEARCH_TERMS if term in path_lower]
            content_terms = []
            content_status = "not_read"
            content = ""
            info = info_by_path.get(member_path)
            is_safe_metadata = record["extension"] in METADATA_EXTENSIONS and record["extension"] not in IMAGE_EXTENSIONS
            if info and not record["is_directory"] and is_safe_metadata and info.file_size <= MAX_METADATA_BYTES:
                try:
                    with archive.open(info, mode="r") as metadata_handle:
                        content = metadata_handle.read(MAX_METADATA_BYTES).decode("utf-8", errors="replace")
                    content_status = "read_from_zip_without_extraction"
                except (OSError, RuntimeError, UnicodeError) as error:
                    content_status = f"unreadable: {error}"
            elif record["extension"] == ".mat":
                content_status = "binary_mat_not_decoded"
            elif record["is_directory"]:
                content_status = "directory_member"
            elif record["extension"] in METADATA_EXTENSIONS:
                content_status = "skipped_over_size_limit"

            if content:
                content_terms = [term for term in SEARCH_TERMS if term in content.lower()]
                for line_number, line in enumerate(content.splitlines()[:MAX_TEXT_LINES_PER_FILE], start=1):
                    line_terms = [term for term in SEARCH_TERMS if term in line.lower()]
                    if line_terms and len(zip_metadata_snippets) < 200:
                        zip_metadata_snippets.append({
                            "member_path": member_path,
                            "line_number": line_number,
                            "terms": ",".join(line_terms),
                            "snippet": re.sub(r"\s+", " ", line.strip())[:240],
                        })

            all_terms = sorted(set(filename_terms + content_terms))
            if all_terms:
                zip_metadata_matches.append({
                    "member_path": member_path,
                    "extension": record["extension"],
                    "filename_terms": ",".join(filename_terms),
                    "content_terms": ",".join(content_terms),
                    "terms": ",".join(all_terms),
                    "content_status": content_status,
                    "evidence_status": "documented" if content_terms else "observed",
                })

            for pattern, label in (
                (r"(?:^|[^a-zA-Z])(?:flight|sequence|session|recording|collection|scene|mission|run|take|trip|set)[-_]?[a-zA-Z0-9]+", "grouping_term_with_value"),
                (r"(?:^|[^a-zA-Z])\d{2,}(?:[^a-zA-Z]|$)", "numeric_identifier"),
                (r"[A-Za-z]+[_-]\d+", "label_number"),
            ):
                matches = re.findall(pattern, member_path, flags=re.IGNORECASE)
                if matches:
                    zip_pattern_counts[label] += len(matches)
                    for match in matches[:3]:
                        if len(zip_pattern_examples[label]) < 10:
                            zip_pattern_examples[label].append(match.strip(" _-"))

            for part in Path(member_path).parts:
                part_lower = part.lower()
                matched_terms = [term for term in GROUPING_TERMS if term in part_lower]
                has_identifier_shape = bool(re.search(r"(?:^|[_-])(?:\d+|[a-z]+\d+)(?:$|[_-])", part_lower))
                if matched_terms or has_identifier_shape:
                    zip_candidate_rows.append({
                        "candidate": part,
                        "source_path": member_path,
                        "matched_terms": ",".join(matched_terms),
                        "evidence_status": "observed",
                        "interpretation": "ZIP path/name pattern only; not proof of an independent flight or session.",
                    })

            parts = list(Path(member_path).parts)
            sensors = {part.lower() for part in parts if part.lower() in {"rgb", "thermal", "visible", "infrared", "ir"}}
            if sensors and not record["is_directory"]:
                sensor = "rgb" if "rgb" in sensors or "visible" in sensors else "thermal"
                canonical_parts = [part.lower() for part in parts if part.lower() not in {"rgb", "thermal", "visible", "infrared", "ir"}]
                canonical = "/".join(canonical_parts[:-1]) if canonical_parts else "."
                existing = next((row for row in zip_sensor_rows if row["canonical_path_without_sensor"] == canonical), None)
                if existing is None:
                    existing = {
                        "canonical_path_without_sensor": canonical,
                        "rgb_filenames": set(),
                        "thermal_filenames": set(),
                    }
                    zip_sensor_rows.append(existing)
                existing[f"{sensor}_filenames"].add(record["filename"].lower())

    for row in zip_sensor_rows:
        rgb_names = row.pop("rgb_filenames")
        thermal_names = row.pop("thermal_filenames")
        row.update({
            "rgb_file_count": len(rgb_names),
            "thermal_file_count": len(thermal_names),
            "shared_filename_count": len(rgb_names & thermal_names),
            "evidence_status": "observed" if rgb_names and thermal_names else "unknown",
            "interpretation": (
                "RGB and thermal occur under a shared canonical ZIP path; verify timestamps/metadata before grouping."
                if rgb_names and thermal_names else
                "No paired ZIP path observed; this does not prove different flights or sessions."
            ),
        })

print(f"ZIP metadata files matching investigation terms: {len(zip_metadata_matches):,}")
print(f"ZIP candidate grouping path/name observations: {len(zip_candidate_rows):,}")
print(f"ZIP RGB/thermal canonical groups: {len(zip_sensor_rows):,}")

ZIP metadata files matching investigation terms: 1
ZIP candidate grouping path/name observations: 5
ZIP RGB/thermal canonical groups: 0


In [52]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def write_csv(filename: str, rows: list[dict], fieldnames: list[str]) -> None:
    output_path = REPORT_DIR / filename
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

extension_rows = [
    {"extension": extension, "file_count": count}
    for extension, count in sorted(extension_counts.items(), key=lambda item: (-item[1], item[0]))
]
directory_extension_rows = [
    {"relative_directory": directory, "extension": extension, "file_count": count}
    for (directory, extension), count in sorted(extension_by_directory.items())
]
directory_rows = [
    {"relative_directory": directory, "file_count": count}
    for directory, count in sorted(directory_file_counts.items())
]
pattern_rows = [
    {"pattern_type": label, "match_count": pattern_counts[label], "examples": "; ".join(pattern_examples[label])}
    for label in sorted(pattern_counts)
]

write_csv("extension_counts.csv", extension_rows, ["extension", "file_count"])
write_csv("directory_extension_counts.csv", directory_extension_rows, ["relative_directory", "extension", "file_count"])
write_csv("directory_file_counts.csv", directory_rows, ["relative_directory", "file_count"])
write_csv("metadata_term_matches.csv", metadata_matches, ["relative_path", "extension", "filename_terms", "content_terms", "terms", "content_status", "evidence_status"])
write_csv("metadata_term_snippets.csv", metadata_snippets, ["relative_path", "line_number", "terms", "snippet"])
write_csv("naming_patterns.csv", pattern_rows, ["pattern_type", "match_count", "examples"])
write_csv("grouping_candidates.csv", candidate_rows[:500], ["candidate", "source_path", "matched_terms", "evidence_status", "interpretation"])
write_csv("rgb_thermal_grouping.csv", sensor_rows[:500], ["canonical_path_without_sensor", "rgb_file_count", "thermal_file_count", "shared_filename_count", "evidence_status", "interpretation"])

(REPORT_DIR / "directory_tree.txt").write_text("\n".join(tree_lines) + "\n", encoding="utf-8")

reported_documented = sum(row["evidence_status"] == "documented" for row in metadata_matches)
reported_observed = sum(row["evidence_status"] == "observed" for row in metadata_matches) + len(candidate_rows)
summary = f"""WiSARv1 organization investigation
=================================
Dataset root: {root_resolved}
Files observed: {len(file_records):,}
Directories observed: {len(directory_records):,}
Metadata-like files observed: {len(metadata_records):,}
Image pixels opened: no
Dataset files modified/copied/moved/extracted: no
Train/validation/test split created: no

Evidence labels
---------------
documented: {reported_documented:,} metadata files contain search terms
observed: {reported_observed:,} filename/content/path observations
unknown: absence of a pattern is not evidence that groups are different flights or sessions

Interpretation limits
---------------------
Names and directory layouts are observations, not proof of independent flights.
A defensible leakage-safe grouping key requires documentation or corroborating metadata such as
explicit flight/session/sequence IDs, timestamps, GPS/trajectory boundaries, or collection records.
RGB/thermal path overlap is reported in rgb_thermal_grouping.csv; shared paths are observed
correspondence, while non-overlap remains unknown.

Reports
-------
{chr(10).join(sorted(path.name for path in REPORT_DIR.iterdir() if path.is_file()))}
"""
(REPORT_DIR / "organization_investigation_report.txt").write_text(summary, encoding="utf-8")

print(summary)


WiSARv1 organization investigation
Dataset root: D:\Hackathons\miniproject5sem\UAV-Search-and-Rescue\data\raw\WiSARD
Files observed: 1
Directories observed: 1
Metadata-like files observed: 0
Image pixels opened: no
Dataset files modified/copied/moved/extracted: no
Train/validation/test split created: no

Evidence labels
---------------
documented: 0 metadata files contain search terms
observed: 0 filename/content/path observations
unknown: absence of a pattern is not evidence that groups are different flights or sessions

Interpretation limits
---------------------
Names and directory layouts are observations, not proof of independent flights.
A defensible leakage-safe grouping key requires documentation or corroborating metadata such as
explicit flight/session/sequence IDs, timestamps, GPS/trajectory boundaries, or collection records.
RGB/thermal path overlap is reported in rgb_thermal_grouping.csv; shared paths are observed
correspondence, while non-overlap remains unknown.

Reports


In [53]:
zip_report_dir = REPORT_DIR
zip_report_dir.mkdir(parents=True, exist_ok=True)

if zip_status == "observed":
    zip_extension_rows = [
        {"extension": extension, "file_count": count}
        for extension, count in sorted(zip_extension_counts.items(), key=lambda item: (-item[1], item[0]))
    ]
    zip_pattern_rows = [
        {"pattern_type": label, "match_count": zip_pattern_counts[label], "examples": "; ".join(zip_pattern_examples[label])}
        for label in sorted(zip_pattern_counts)
    ]
    write_csv("zip_extension_counts.csv", zip_extension_rows, ["extension", "file_count"])
    write_csv("zip_metadata_term_matches.csv", zip_metadata_matches, ["member_path", "extension", "filename_terms", "content_terms", "terms", "content_status", "evidence_status"])
    write_csv("zip_metadata_term_snippets.csv", zip_metadata_snippets, ["member_path", "line_number", "terms", "snippet"])
    write_csv("zip_naming_patterns.csv", zip_pattern_rows, ["pattern_type", "match_count", "examples"])
    write_csv("zip_grouping_candidates.csv", zip_candidate_rows[:500], ["candidate", "source_path", "matched_terms", "evidence_status", "interpretation"])
    write_csv("zip_rgb_thermal_grouping.csv", zip_sensor_rows[:500], ["canonical_path_without_sensor", "rgb_file_count", "thermal_file_count", "shared_filename_count", "evidence_status", "interpretation"])
    (zip_report_dir / "zip_directory_tree.txt").write_text("\n".join(zip_tree_lines) + "\n", encoding="utf-8")

    zip_documented = sum(row["evidence_status"] == "documented" for row in zip_metadata_matches)
    zip_observed = sum(row["evidence_status"] == "observed" for row in zip_metadata_matches) + len(zip_candidate_rows)
    zip_summary = f"""WiSARDv1 ZIP organization investigation
=======================================
ZIP path: {zip_path}
ZIP members observed: {len(zip_member_records):,}
Image pixels opened: no
ZIP extracted/copied/moved/modified: no
Train/validation/test split created: no

Evidence labels
---------------
documented: {zip_documented:,} ZIP metadata/path records with metadata text terms
observed: {zip_observed:,} ZIP filename/content/path observations
unknown: different ZIP folder names are not treated as different flights without documentation or metadata

Metadata reads
--------------
Only non-image metadata-like members at or below MAX_METADATA_BYTES were read with ZipFile.open().
No image member was opened, decoded, resized, or extracted.

RGB/thermal
-----------
Shared canonical ZIP paths are labeled observed correspondence only.
Non-overlap is labeled unknown, not evidence of different flights or sessions.
"""
    (zip_report_dir / "zip_organization_investigation_report.txt").write_text(zip_summary, encoding="utf-8")
    print(zip_summary)
else:
    print("No WiSARDv1.zip detected under DATASET_ROOT; ZIP reports were not written.")


No WiSARDv1.zip detected under DATASET_ROOT; ZIP reports were not written.


## Collection-context grouping analysis

This section is a reproducible, read-only grouping candidate for later leakage control. A recording name is parsed only when it has the form `date_location_platform_modality_identifier`, where `date` is six digits, `modality` is exactly `VIS` or `IR`, and `identifier` is numeric. The candidate collection context is the exact token prefix before the modality token. For example, `210417_MtErie_Enterprise_VIS_0003` maps to `210417_MtErie_Enterprise`. Names that do not satisfy this rule are retained in the ambiguity report and are never silently merged.

These are observed naming/path relationships, not documented flight identities. No split assignment is made here.

In [54]:
ANNOTATION_EXTENSIONS = {
    ".txt", ".xml", ".json", ".csv", ".tsv", ".mat", ".yaml", ".yml",
    ".ann", ".label", ".labels",
}
MODALITY_TOKENS = {"VIS": "VIS", "IR": "IR"}
FRAME_INDEX_PATTERNS = (
    re.compile(r"(?i)(?:frame|image|img|rgb|ir|thermal)[_-]?(\d+)(?:\D*)$"),
    re.compile(r"(?:^|[_-])(\d{3,})(?:\D*)$"),
)


def parse_recording_name(recording_name: str) -> dict:
    """Parse one folder name using the explicit date/prefix/modality/id rule."""
    tokens = recording_name.split("_")
    modality_positions = [index for index, token in enumerate(tokens) if token.upper() in MODALITY_TOKENS]
    result = {
        "recording_name": recording_name,
        "collection_context": "",
        "modality": "",
        "date_token": tokens[0] if tokens and re.fullmatch(r"\d{6}", tokens[0]) else "",
        "location_token": "",
        "platform_token": "",
        "stream_identifier": tokens[-1] if tokens and tokens[-1].isdigit() else "",
        "parsing_status": "ambiguous_or_unparsed",
    }
    if len(modality_positions) != 1:
        return result

    modality_index = modality_positions[0]
    has_date = bool(re.fullmatch(r"\d{6}", tokens[0])) if tokens else False
    has_prefix = modality_index >= 2
    has_numeric_identifier = bool(tokens) and bool(re.fullmatch(r"\d+", tokens[-1])) and modality_index == len(tokens) - 2
    if not (has_date and has_prefix and has_numeric_identifier):
        return result

    prefix_tokens = tokens[:modality_index]
    result.update({
        "collection_context": "_".join(prefix_tokens),
        "modality": MODALITY_TOKENS[tokens[modality_index].upper()],
        "location_token": "_".join(prefix_tokens[1:-1]),
        "platform_token": prefix_tokens[-1],
        "parsing_status": "parsed",
    })
    return result


def top_level_recording_component(member_path: str) -> tuple[str, str]:
    """Choose the first parseable directory component, preserving its original path/name."""
    parts = Path(member_path).parts
    directory_parts = parts[:-1]
    for index, part in enumerate(directory_parts):
        if parse_recording_name(part)["parsing_status"] == "parsed":
            return part, "/".join(directory_parts[: index + 1])
    if directory_parts:
        return directory_parts[0], directory_parts[0]
    return "", ""


def frame_index_from_name(filename: str) -> int | None:
    stem = Path(filename).stem
    for pattern in FRAME_INDEX_PATTERNS:
        match = pattern.search(stem)
        if match:
            return int(match.group(1))
    return None


def is_image_member(record: dict) -> bool:
    return not record["is_directory"] and record["extension"] in IMAGE_EXTENSIONS


def is_annotation_member(record: dict) -> bool:
    return not record["is_directory"] and record["extension"] in ANNOTATION_EXTENSIONS


collection_context_summary = []
recording_stream_summary = []
collection_context_modalities = []
recording_sequence_structure = []
collection_context_sanity_checks = []
collection_context_ambiguities = []

if zip_status == "observed":
    recording_aggregates = {}
    relative_path_counts = Counter()
    recording_name_to_contexts = defaultdict(set)
    context_to_recording_names = defaultdict(set)
    context_metadata = {}

    for record in sorted(zip_member_records, key=lambda item: item["member_path"]):
        relative_path = record["member_path"]
        relative_path_counts[relative_path] += 1
        if record["is_directory"]:
            continue
        recording_name, recording_path = top_level_recording_component(relative_path)
        if not recording_name:
            collection_context_ambiguities.append({
                "recording_name": "",
                "source_path": relative_path,
                "reason": "no_directory_component",
                "evidence_status": "observed",
                "interpretation": "No recording folder can be derived from this member path.",
            })
            continue

        parsed = parse_recording_name(recording_name)
        aggregate = recording_aggregates.setdefault(recording_path, {
            **parsed,
            "recording_name": recording_name,
            "recording_path": recording_path,
            "image_count": 0,
            "annotation_count": 0,
            "image_extensions": Counter(),
            "annotation_extensions": Counter(),
            "example_frame_names": [],
            "frame_indices": set(),
            "all_frame_indices_parseable": True,
        })
        if is_image_member(record):
            aggregate["image_count"] += 1
            aggregate["image_extensions"][record["extension"]] += 1
            if len(aggregate["example_frame_names"]) < 5:
                aggregate["example_frame_names"].append(Path(relative_path).name)
            frame_index = frame_index_from_name(Path(relative_path).name)
            if frame_index is None:
                aggregate["all_frame_indices_parseable"] = False
            else:
                aggregate["frame_indices"].add(frame_index)
        elif is_annotation_member(record):
            aggregate["annotation_count"] += 1
            aggregate["annotation_extensions"][record["extension"]] += 1

        if aggregate["parsing_status"] == "parsed":
            context = aggregate["collection_context"]
            recording_name_to_contexts[recording_name].add(context)
            context_to_recording_names[context].add(recording_name)
            context_metadata.setdefault(context, aggregate)
        else:
            collection_context_ambiguities.append({
                "recording_name": recording_name,
                "source_path": relative_path,
                "reason": "recording_name_did_not_match_explicit_parser",
                "evidence_status": "observed",
                "interpretation": "Retained as ambiguous; no collection context was assigned.",
            })

    for recording_path, aggregate in sorted(recording_aggregates.items()):
        parsed = aggregate["parsing_status"] == "parsed"
        frame_indices = aggregate["frame_indices"]
        min_frame = min(frame_indices) if frame_indices and aggregate["all_frame_indices_parseable"] else ""
        max_frame = max(frame_indices) if frame_indices and aggregate["all_frame_indices_parseable"] else ""
        gaps = []
        if frame_indices and aggregate["all_frame_indices_parseable"]:
            gaps = sorted(set(range(min_frame, max_frame + 1)) - frame_indices)
        recording_stream_summary.append({
            "recording_name": aggregate["recording_name"],
            "collection_context": aggregate["collection_context"],
            "modality": aggregate["modality"],
            "date_token": aggregate["date_token"],
            "location_token": aggregate["location_token"],
            "platform_token": aggregate["platform_token"],
            "stream_identifier": aggregate["stream_identifier"],
            "image_count": aggregate["image_count"],
            "annotation_count": aggregate["annotation_count"],
            "image_extensions": ";".join(sorted(aggregate["image_extensions"])),
            "annotation_extensions": ";".join(sorted(aggregate["annotation_extensions"])),
            "example_frame_names": ";".join(sorted(aggregate["example_frame_names"])),
            "evidence_status": "observed",
            "interpretation": "Recording identity and modality are derived from ZIP path/name structure only; not a documented flight.",
        })
        recording_sequence_structure.append({
            "recording_name": aggregate["recording_name"],
            "collection_context": aggregate["collection_context"],
            "modality": aggregate["modality"],
            "frame_index_parse_status": "reliably_parseable" if aggregate["all_frame_indices_parseable"] and frame_indices else "unknown_or_mixed",
            "minimum_frame_index": min_frame,
            "maximum_frame_index": max_frame,
            "unique_frame_index_count": len(frame_indices),
            "obvious_gap_count": len(gaps),
            "obvious_gap_examples": ";".join(str(value) for value in gaps[:20]),
            "evidence_status": "observed",
            "interpretation": "Frame sequence is inferred from filename patterns only; no temporal synchronization is claimed.",
        })

    context_aggregates = defaultdict(lambda: {
        "recordings": [], "VIS": [], "IR": [], "visual_images": 0, "thermal_images": 0,
        "annotation_count": 0, "dates": set(), "locations": set(), "platforms": set(), "identifiers": set(),
    })
    for aggregate in recording_aggregates.values():
        if aggregate["parsing_status"] != "parsed":
            continue
        context = context_aggregates[aggregate["collection_context"]]
        context["recordings"].append(aggregate)
        context[aggregate["modality"]].append(aggregate)
        context["visual_images"] += aggregate["image_count"] if aggregate["modality"] == "VIS" else 0
        context["thermal_images"] += aggregate["image_count"] if aggregate["modality"] == "IR" else 0
        context["annotation_count"] += aggregate["annotation_count"]
        context["dates"].add(aggregate["date_token"])
        context["locations"].add(aggregate["location_token"])
        context["platforms"].add(aggregate["platform_token"])
        context["identifiers"].add(aggregate["stream_identifier"])

    for context_name in sorted(context_aggregates):
        context = context_aggregates[context_name]
        modalities = sorted({aggregate["modality"] for aggregate in context["recordings"]})
        collection_context_summary.append({
            "collection_context": context_name,
            "top_level_recording_count": len(context["recordings"]),
            "visual_recording_count": len(context["VIS"]),
            "thermal_recording_count": len(context["IR"]),
            "visual_image_count": context["visual_images"],
            "thermal_image_count": context["thermal_images"],
            "annotation_count": context["annotation_count"],
            "recording_names": ";".join(sorted(aggregate["recording_name"] for aggregate in context["recordings"])),
            "modality_set": ";".join(modalities),
            "date_token": ";".join(sorted(value for value in context["dates"] if value)),
            "location_token": ";".join(sorted(value for value in context["locations"] if value)),
            "platform_token": ";".join(sorted(value for value in context["platforms"] if value)),
            "identifier_tokens": ";".join(sorted(value for value in context["identifiers"] if value)),
            "parsing_status": "parsed",
            "evidence_status": "observed",
            "interpretation": "Candidate collection context from shared name prefix; not evidence of an independent flight.",
        })
        collection_context_modalities.append({
            "collection_context": context_name,
            "contains_both_modalities": "yes" if context["VIS"] and context["IR"] else "no",
            "visual_stream_count": len(context["VIS"]),
            "thermal_stream_count": len(context["IR"]),
            "visual_image_count": context["visual_images"],
            "thermal_image_count": context["thermal_images"],
            "evidence_status": "observed",
            "interpretation": "VIS/IR coexistence is an observed path relationship; matching identifiers do not prove synchronization.",
        })

    for recording_name, contexts in sorted(recording_name_to_contexts.items()):
        if len(contexts) > 1:
            collection_context_sanity_checks.append({
                "check_type": "recording_in_multiple_contexts",
                "status": "flagged",
                "details": f"{recording_name} -> {';'.join(sorted(contexts))}",
                "evidence_status": "observed",
                "interpretation": "Parser ambiguity or duplicate naming requires review.",
            })

    duplicate_recording_names = Counter(aggregate["recording_name"] for aggregate in recording_aggregates.values())
    for recording_name, count in sorted(duplicate_recording_names.items()):
        if count > 1:
            collection_context_sanity_checks.append({
                "check_type": "duplicate_recording_name",
                "status": "flagged",
                "details": f"{recording_name} occurs {count} times",
                "evidence_status": "observed",
                "interpretation": "Duplicate folder names occur at multiple paths; retain full paths for auditing.",
            })

    for relative_path, count in sorted(relative_path_counts.items()):
        if count > 1:
            collection_context_sanity_checks.append({
                "check_type": "duplicate_relative_path",
                "status": "flagged",
                "details": f"{relative_path} occurs {count} times in ZIP central directory",
                "evidence_status": "observed",
                "interpretation": "Duplicate archive members require review before any future grouping use.",
            })

    for context_name, recording_names in sorted(context_to_recording_names.items()):
        modalities = {parse_recording_name(name)["modality"] for name in recording_names}
        if modalities == {"VIS", "IR"}:
            collection_context_sanity_checks.append({
                "check_type": "context_contains_both_modalities",
                "status": "observed",
                "details": f"{context_name} -> VIS and IR",
                "evidence_status": "observed",
                "interpretation": "Shared candidate context is a leakage-control candidate, not a documented flight.",
            })

    context_names = sorted(context_aggregates)
    for left_index, left_name in enumerate(context_names):
        for right_name in context_names[left_index + 1:]:
            left_base = re.sub(r"(?:_VIS|_IR)(?:_|$)", "_", left_name, flags=re.IGNORECASE)
            right_base = re.sub(r"(?:_VIS|_IR)(?:_|$)", "_", right_name, flags=re.IGNORECASE)
            if left_base == right_base or left_name.rstrip("_0123456789") == right_name.rstrip("_0123456789"):
                collection_context_sanity_checks.append({
                    "check_type": "similar_context_names",
                    "status": "flagged",
                    "details": f"{left_name} ~ {right_name}",
                    "evidence_status": "observed",
                    "interpretation": "Similar names need review; the parser does not merge them automatically.",
                })

    ambiguous_count = len(collection_context_ambiguities)
    print(f"Candidate collection contexts: {len(collection_context_summary):,}")
    print(f"VIS recordings: {sum(row['visual_recording_count'] for row in collection_context_summary):,}")
    print(f"IR recordings: {sum(row['thermal_recording_count'] for row in collection_context_summary):,}")
    print(f"Contexts containing both modalities: {sum(row['contains_both_modalities'] == 'yes' for row in collection_context_modalities):,}")
    print(f"Ambiguous/unparsed recordings: {ambiguous_count:,}")
    print("No train/validation/test split created.")
else:
    print("ZIP collection-context analysis skipped because WiSARDv1.zip was not detected.")

ZIP collection-context analysis skipped because WiSARDv1.zip was not detected.


In [55]:
if zip_status == "observed":
    unique_ambiguities = {}
    for row in collection_context_ambiguities:
        source_directory = row["source_path"].rsplit("/", 1)[0] if "/" in row["source_path"] else row["source_path"]
        key = (row["recording_name"], source_directory, row["reason"])
        unique_ambiguities[key] = row
    collection_context_ambiguities = [unique_ambiguities[key] for key in sorted(unique_ambiguities)]


In [56]:
if zip_status == "observed":
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    extension_counts_by_recording = defaultdict(Counter)
    recording_path_by_name = defaultdict(set)
    for record in zip_member_records:
        if record["is_directory"]:
            continue
        recording_name, recording_path = top_level_recording_component(record["member_path"])
        if recording_name:
            extension_counts_by_recording[recording_path][record["extension"]] += 1
            recording_path_by_name[recording_name].add(recording_path)

    for row in recording_stream_summary:
        paths = recording_path_by_name.get(row["recording_name"], set())
        extension_counter = Counter()
        for path in paths:
            extension_counter.update(extension_counts_by_recording[path])
        row["file_extension_counts"] = ";".join(
            f"{extension}={count}" for extension, count in sorted(extension_counter.items())
        )

    context_extension_counts = defaultdict(Counter)
    for aggregate in recording_aggregates.values():
        if aggregate["parsing_status"] == "parsed":
            context_extension_counts[aggregate["collection_context"]].update(
                extension_counts_by_recording[aggregate["recording_path"]]
            )
    for row in collection_context_summary:
        row["file_extension_counts"] = ";".join(
            f"{extension}={count}"
            for extension, count in sorted(context_extension_counts[row["collection_context"]].items())
        )

    write_csv(
        "collection_context_summary.csv",
        collection_context_summary,
        [
            "collection_context", "top_level_recording_count", "visual_recording_count",
            "thermal_recording_count", "visual_image_count", "thermal_image_count",
            "annotation_count", "recording_names", "modality_set", "date_token",
            "location_token", "platform_token", "identifier_tokens", "file_extension_counts",
            "parsing_status", "evidence_status", "interpretation",
        ],
    )
    write_csv(
        "recording_stream_summary.csv",
        recording_stream_summary,
        [
            "recording_name", "collection_context", "modality", "date_token", "location_token",
            "platform_token", "stream_identifier", "image_count", "annotation_count",
            "image_extensions", "annotation_extensions", "file_extension_counts",
            "example_frame_names", "evidence_status", "interpretation",
        ],
    )
    write_csv(
        "collection_context_modalities.csv",
        collection_context_modalities,
        [
            "collection_context", "contains_both_modalities", "visual_stream_count",
            "thermal_stream_count", "visual_image_count", "thermal_image_count",
            "evidence_status", "interpretation",
        ],
    )
    write_csv(
        "recording_sequence_structure.csv",
        recording_sequence_structure,
        [
            "recording_name", "collection_context", "modality", "frame_index_parse_status",
            "minimum_frame_index", "maximum_frame_index", "unique_frame_index_count",
            "obvious_gap_count", "obvious_gap_examples", "evidence_status", "interpretation",
        ],
    )
    write_csv(
        "collection_context_ambiguities.csv",
        collection_context_ambiguities,
        ["recording_name", "source_path", "reason", "evidence_status", "interpretation"],
    )
    write_csv(
        "collection_context_sanity_checks.csv",
        collection_context_sanity_checks,
        ["check_type", "status", "details", "evidence_status", "interpretation"],
    )

    both_modality_contexts = [
        row["collection_context"]
        for row in collection_context_modalities
        if row["contains_both_modalities"] == "yes"
    ]
    context_lines = []
    for row in collection_context_summary:
        context_lines.append(
            f"- {row['collection_context']}: {row['visual_recording_count']} VIS, "
            f"{row['thermal_recording_count']} IR, modalities={row['modality_set']}"
        )
    collection_context_report = f"""WiSARDv1 collection-context grouping analysis
===============================================
ZIP path: {zip_path}

Parsing rule
------------
A recording folder is parsed only when its exact underscore-separated name has one six-digit
first token, exactly one VIS or IR token, and a numeric final token immediately after that
modality token. The candidate collection_context is the exact original token prefix before
VIS/IR. The token immediately before VIS/IR is reported as platform_token; tokens between
the date and platform are reported as location_token. These labels describe name structure
only and do not assert the semantic meaning of any token.

Results
-------
Candidate collection contexts: {len(collection_context_summary):,}
VIS recordings: {sum(row['visual_recording_count'] for row in collection_context_summary):,}
IR recordings: {sum(row['thermal_recording_count'] for row in collection_context_summary):,}
Contexts containing both modalities: {len(both_modality_contexts):,}
Ambiguous/unparsed recordings: {len(collection_context_ambiguities):,}

Contexts
--------
{chr(10).join(context_lines) if context_lines else '- none observed'}

Interpretation and limits
-------------------------
All collection contexts and modality relationships are observed from ZIP path/name structure.
They are methodological leakage-control candidates, not documented independent flights.
No folder-name difference is treated as proof of a different flight. Matching suffix numbers
do not establish temporal synchronization or paired frames. Frame ranges and gaps are inferred
from image filenames only; image pixels were never opened or decoded.

Sanity checks
-------------
See collection_context_sanity_checks.csv for duplicate paths/names, multi-context names,
contexts containing both VIS and IR, and suspiciously similar context names.
See collection_context_ambiguities.csv for names that did not satisfy the parser and were not
assigned to a candidate context.

Safety and reproducibility
--------------------------
ZIP central-directory metadata was inspected directly with zipfile.ZipFile. No extraction,
copying, moving, renaming, resizing, image processing, model training, or split assignment
was performed. Outputs are sorted deterministically by path/name. No train/validation/test
split has been created.
"""
    (REPORT_DIR / "collection_context_analysis_report.txt").write_text(collection_context_report, encoding="utf-8")
    print(collection_context_report)
else:
    print("Collection-context reports skipped because WiSARDv1.zip was not detected.")

Collection-context reports skipped because WiSARDv1.zip was not detected.


In [57]:
parser_examples = {
    "210417_MtErie_Enterprise_VIS_0003": "210417_MtErie_Enterprise",
    "210417_MtErie_Enterprise_IR_0004": "210417_MtErie_Enterprise",
    "210327_Airfield_FLIR_VIS_1": "210327_Airfield_FLIR",
    "210327_Airfield_FLIR_IR_1": "210327_Airfield_FLIR",
}
for example_name, expected_context in parser_examples.items():
    parsed_example = parse_recording_name(example_name)
    assert parsed_example["collection_context"] == expected_context
    assert parsed_example["parsing_status"] == "parsed"
print(f"Parser self-check passed for {len(parser_examples)} supplied naming examples.")


Parser self-check passed for 4 supplied naming examples.


In [58]:
if zip_status == "observed":
    print("\nFinal collection-context summary")
    print(f"Candidate collection contexts: {len(collection_context_summary)}")
    print(f"VIS recordings: {sum(row['visual_recording_count'] for row in collection_context_summary)}")
    print(f"IR recordings: {sum(row['thermal_recording_count'] for row in collection_context_summary)}")
    print(f"Contexts containing both modalities: {sum(row['contains_both_modalities'] == 'yes' for row in collection_context_modalities)}")
    print(f"Ambiguous/unparsed recordings: {len(collection_context_ambiguities)}")
    print("No train/validation/test split created.")
else:
    print("\nFinal collection-context summary unavailable: WiSARDv1.zip was not detected.")



Final collection-context summary unavailable: WiSARDv1.zip was not detected.
